
# Feed-Forward Neural Network — AQI Forecasting

This notebook implements a **tabular feed-forward neural network (FNN / MLP)** for the India AQI regression task.
It follows the project's shared preprocessing pipeline and uses a **temporal train/validation/test split** to avoid leakage.

**Why this model?**
- Satisfies the course requirement to include at least one neural-network-based method.
- Fits the project's tabular feature set directly.
- Extends the simple baseline with a nonlinear learner over pollutants, missingness flags, calendar features, lag/rolling AQI features, and city dummies.



## Build plan

1. Load the shared AQI dataset and reuse `preprocessing.py` so this notebook stays consistent with the baseline and comparison notebooks.
2. Build the full engineered tabular feature matrix already defined for the project.
3. Split data **temporally** into train / validation / test.
4. Standardize features using statistics from the training subset only.
5. Train a PyTorch feed-forward neural network on `log1p(AQI)`.
6. Tune a small set of hyperparameters using validation RMSE on the original AQI scale.
7. Retrain/evaluate the selected architecture and save metrics to `results/fnn_metrics.json` for `comparison.ipynb`.


In [ ]:

import copy
import os
import random

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

import torch
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error

from preprocessing import (
    load_and_prepare, impute, build_features,
    get_arrays, evaluate, save_results,
    CUTOFF_DATE,
)

plt.style.use('seaborn-v0_8-darkgrid')
%matplotlib inline

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'device: {DEVICE}')


## 1. Load data and create tabular features

In [ ]:

df = load_and_prepare()
df = impute(df)
df, feature_cols = build_features(df)

print(f'rows after preprocessing: {len(df):,}')
print(f'number of tabular features: {len(feature_cols)}')
print('sample features:')
print(feature_cols[:15])



## 2. Temporal split

We keep the project's test split fixed at **2019-12-01** and carve a validation block out of the training period.
This mirrors the LSTM notebook's leakage-safe workflow.


In [ ]:

VAL_CUTOFF = pd.Timestamp('2019-10-01')

full_train_df = df[df['date'] < CUTOFF_DATE].copy().reset_index(drop=True)
test_df       = df[df['date'] >= CUTOFF_DATE].copy().reset_index(drop=True)

train_df = full_train_df[full_train_df['date'] < VAL_CUTOFF].copy().reset_index(drop=True)
val_df   = full_train_df[full_train_df['date'] >= VAL_CUTOFF].copy().reset_index(drop=True)

print(f'train rows: {len(train_df):,}')
print(f'val rows  : {len(val_df):,}')
print(f'test rows : {len(test_df):,}')
print(f'train date range: {train_df["date"].min().date()} to {train_df["date"].max().date()}')
print(f'val date range  : {val_df["date"].min().date()} to {val_df["date"].max().date()}')
print(f'test date range : {test_df["date"].min().date()} to {test_df["date"].max().date()}')


## 3. Build arrays and standardize inputs

In [ ]:

X_train, X_val, y_train_log, y_val_raw, y_val_log = get_arrays(train_df, val_df, feature_cols)
_, X_test, _, y_test_raw, y_test_log = get_arrays(train_df, test_df, feature_cols)

scaler = StandardScaler()
X_train_s = scaler.fit_transform(X_train).astype(np.float32)
X_val_s   = scaler.transform(X_val).astype(np.float32)
X_test_s  = scaler.transform(X_test).astype(np.float32)

y_train_t = y_train_log.astype(np.float32).reshape(-1, 1)
y_val_t   = y_val_log.astype(np.float32).reshape(-1, 1)
y_test_t  = y_test_log.astype(np.float32).reshape(-1, 1)

print('scaled shapes:')
print('X_train:', X_train_s.shape, 'X_val:', X_val_s.shape, 'X_test:', X_test_s.shape)


## 4. Dataloaders

In [ ]:

def make_loader(X, y, batch_size=256, shuffle=False):
    ds = TensorDataset(
        torch.tensor(X, dtype=torch.float32),
        torch.tensor(y, dtype=torch.float32),
    )
    return DataLoader(ds, batch_size=batch_size, shuffle=shuffle)

train_loader = make_loader(X_train_s, y_train_t, batch_size=256, shuffle=True)
val_loader   = make_loader(X_val_s, y_val_t, batch_size=512, shuffle=False)
test_loader  = make_loader(X_test_s, y_test_t, batch_size=512, shuffle=False)

len(train_loader), len(val_loader), len(test_loader)



## 5. Model definition

This is a standard multilayer perceptron for tabular regression:

`input → Linear → ReLU → BatchNorm → Dropout → ... → Linear(1)`


In [ ]:

class AQIFeedForwardNN(nn.Module):
    def __init__(self, input_dim, hidden_dims=(128, 64), dropout=0.2):
        super().__init__()
        layers = []
        prev = input_dim
        for h in hidden_dims:
            layers.extend([
                nn.Linear(prev, h),
                nn.ReLU(),
                nn.BatchNorm1d(h),
                nn.Dropout(dropout),
            ])
            prev = h
        layers.append(nn.Linear(prev, 1))
        self.net = nn.Sequential(*layers)

    def forward(self, x):
        return self.net(x)

model = AQIFeedForwardNN(input_dim=X_train_s.shape[1])
print(model)


## 6. Training utilities

In [ ]:

def predict_log(model, loader, device='cpu'):
    model.eval()
    preds = []
    with torch.no_grad():
        for X_b, _ in loader:
            X_b = X_b.to(device)
            preds.append(model(X_b).cpu().numpy())
    return np.vstack(preds).reshape(-1)


def train_model(model, train_loader, val_loader, val_raw_aqi,
                epochs=150, lr=1e-3, weight_decay=1e-5,
                patience=15, device='cpu', verbose=True):
    model = model.to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=weight_decay)
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
        optimizer, mode='min', factor=0.5, patience=5
    )
    criterion = nn.MSELoss()

    best_state = None
    best_rmse = float('inf')
    wait = 0

    history = {
        'train_loss': [],
        'val_rmse': [],
    }

    for epoch in range(1, epochs + 1):
        model.train()
        batch_losses = []

        for X_b, y_b in train_loader:
            X_b = X_b.to(device)
            y_b = y_b.to(device)

            optimizer.zero_grad()
            pred = model(X_b)
            loss = criterion(pred, y_b)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()
            batch_losses.append(loss.item())

        train_loss = float(np.mean(batch_losses))

        val_pred_log = predict_log(model, val_loader, device=device)
        val_pred_raw = np.clip(np.expm1(val_pred_log), 0, None)
        val_rmse = float(np.sqrt(mean_squared_error(val_raw_aqi, val_pred_raw)))

        history['train_loss'].append(train_loss)
        history['val_rmse'].append(val_rmse)

        scheduler.step(val_rmse)

        improved = val_rmse < best_rmse
        if improved:
            best_rmse = val_rmse
            best_state = copy.deepcopy(model.state_dict())
            wait = 0
        else:
            wait += 1

        if verbose and (epoch == 1 or epoch % 10 == 0):
            print(f'epoch {epoch:>3d} | train loss {train_loss:.4f} | val RMSE {val_rmse:.2f}')

        if wait >= patience:
            if verbose:
                print(f'early stopping at epoch {epoch}')
            break

    model.load_state_dict(best_state)
    return model, history, best_rmse


## 7. Hyperparameter search

In [ ]:

search_space = [
    {'hidden_dims': (128, 64), 'dropout': 0.20, 'lr': 1e-3, 'weight_decay': 1e-5},
    {'hidden_dims': (256, 128), 'dropout': 0.20, 'lr': 1e-3, 'weight_decay': 1e-5},
    {'hidden_dims': (128, 64, 32), 'dropout': 0.30, 'lr': 5e-4, 'weight_decay': 1e-4},
]

search_results = []

for i, cfg in enumerate(search_space, start=1):
    print(f'
>>> config {i}: {cfg}')
    candidate = AQIFeedForwardNN(
        input_dim=X_train_s.shape[1],
        hidden_dims=cfg['hidden_dims'],
        dropout=cfg['dropout'],
    )
    candidate, history, best_rmse = train_model(
        candidate,
        train_loader,
        val_loader,
        y_val_raw,
        epochs=120,
        lr=cfg['lr'],
        weight_decay=cfg['weight_decay'],
        patience=12,
        device=DEVICE,
        verbose=False,
    )
    record = dict(cfg)
    record['best_val_rmse'] = best_rmse
    record['epochs_ran'] = len(history['train_loss'])
    search_results.append(record)
    print(f"best val RMSE: {best_rmse:.2f} | epochs: {record['epochs_ran']}")

search_df = pd.DataFrame(search_results).sort_values('best_val_rmse').reset_index(drop=True)
search_df


## 8. Train selected configuration and inspect learning curves

In [ ]:

best_cfg = search_df.iloc[0].to_dict()
print('selected config:')
print(best_cfg)

best_model = AQIFeedForwardNN(
    input_dim=X_train_s.shape[1],
    hidden_dims=tuple(best_cfg['hidden_dims']),
    dropout=float(best_cfg['dropout']),
)

best_model, history, best_val_rmse = train_model(
    best_model,
    train_loader,
    val_loader,
    y_val_raw,
    epochs=150,
    lr=float(best_cfg['lr']),
    weight_decay=float(best_cfg['weight_decay']),
    patience=15,
    device=DEVICE,
    verbose=True,
)

print(f'best validation RMSE: {best_val_rmse:.2f}')


In [ ]:

fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(history['train_loss'], label='train loss')
ax.set_title('Training loss by epoch')
ax.set_xlabel('epoch')
ax.set_ylabel('MSE loss on log1p(AQI)')
ax.legend()
plt.tight_layout()
plt.show()

fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(history['val_rmse'], label='validation RMSE')
ax.set_title('Validation RMSE by epoch')
ax.set_xlabel('epoch')
ax.set_ylabel('RMSE on original AQI scale')
ax.legend()
plt.tight_layout()
plt.show()


## 9. Evaluate on the held-out test set

In [ ]:

test_pred_log = predict_log(best_model, test_loader, device=DEVICE)
test_pred_raw = np.clip(np.expm1(test_pred_log), 0, None)

label = f"FeedForwardNN {tuple(best_cfg['hidden_dims'])}, dropout={best_cfg['dropout']}, lr={best_cfg['lr']}"
result = evaluate(y_test_raw, test_pred_raw, label=label)
save_results(result, 'fnn_tabular')


## 10. Predicted vs actual on the test set

In [ ]:

plot_df = pd.DataFrame({
    'actual_aqi': y_test_raw,
    'predicted_aqi': test_pred_raw,
})

fig, ax = plt.subplots(figsize=(6, 6))
ax.scatter(plot_df['actual_aqi'], plot_df['predicted_aqi'], alpha=0.25)
lims = [0, max(plot_df['actual_aqi'].max(), plot_df['predicted_aqi'].max())]
ax.plot(lims, lims, linestyle='--')
ax.set_xlim(lims)
ax.set_ylim(lims)
ax.set_xlabel('Actual AQI')
ax.set_ylabel('Predicted AQI')
ax.set_title('Feed-forward NN: actual vs predicted AQI')
plt.tight_layout()
plt.show()



## 11. Next step

Run `comparison.ipynb` after this notebook to compare the neural network against the baseline, k-NN, tree models, and LSTM using the saved JSON metrics file.
